以下使用CLIP的zero-shot进行图像分类,待分类图片为狗狗的照片

In [ ]:
# Zero-shot 诊断：dog.jpg 在“几个动物类别”上的可判定性
# 依赖你前面已经有 model / processor

import torch
import numpy as np
import torch.nn.functional as F
from PIL import Image

image_path = "./pictures/dog.jpg"
image = Image.open(image_path).convert("RGB")

# 只列几个动物类别
labels = ["狗", "猫", "兔子", "马", "鸟"]

# 多模板（看稳定性）
templates = [
    "这是一张{}的照片",
    "图片中是{}",
    "这只动物是{}",
]

device = next(model.parameters()).device

all_probs = []
for t in templates:
    text_prompts = [t.format(x) for x in labels]

    inputs = processor(
        text=text_prompts,
        images=image,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits_per_image, dim=1).cpu().numpy()[0]  # shape: [num_labels]
    all_probs.append(probs)

all_probs = np.stack(all_probs, axis=0)      # [num_templates, num_labels]
mean_probs = all_probs.mean(axis=0)          # 融合后概率
std_probs = all_probs.std(axis=0)            # 模板敏感度（越小越稳定）

order = np.argsort(-mean_probs)
top1, top2 = order[0], order[1]

print(f"Image: {image_path}\n")
print("=== 平均概率（跨模板）===")
for i in order:
    print(f"{labels[i]:<6} prob={mean_probs[i]:.4f}  std={std_probs[i]:.4f}")

# 置信度指标
margin = float(mean_probs[top1] - mean_probs[top2])  # top1-top2间隔
entropy = float(-(mean_probs * np.log(mean_probs + 1e-12)).sum())  # 越小越确定
consistency = float((all_probs.argmax(axis=1) == top1).mean())      # 模板一致率

print("\n=== 诊断指标 ===")
print(f"预测类别: {labels[top1]}")
print(f"top1-top2 margin: {margin:.4f}   (越大越稳)")
print(f"entropy: {entropy:.4f}           (越小越稳)")
print(f"template consistency: {consistency:.2%} (越高越稳)")

此处依赖本地下载的model,以及先前做的processor工作,这里略去,直接给出运行结果,结果为:

In [ ]:
Image: ./pictures/dog.jpg

=== 平均概率（跨模板）===
狗      prob=0.9687  std=0.0273
马      prob=0.0138  std=0.0138
兔子     prob=0.0074  std=0.0044
猫      prob=0.0074  std=0.0059
鸟      prob=0.0027  std=0.0031

=== 诊断指标 ===
预测类别: 狗
top1-top2 margin: 0.9549   (越大越稳)
entropy: 0.1783           (越小越稳)
template consistency: 100.00% (越高越稳)

可以看到还是比较稳的,原始狗狗图片为:
![dog](dog.jpg)